In [15]:
%%writefile RulesForIntents.py

import os
import re
import io
import email
import time
import json
import torch
import shutil
import imaplib
import smtplib
import tempfile
import pandas as pd
import fitz  
from PIL import Image
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from email.header import decode_header
from email.mime.text import MIMEText
from hijri_converter import Hijri
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers import BM25Retriever, ParentDocumentRetriever
from langchain.storage import InMemoryStore
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from googleapiclient.discovery import build
from google.oauth2 import service_account
from ArabicOcr import arabicocr  
import gc  
import openpyxl  
import json
import re
from datetime import datetime, timedelta
from hijri_converter import Hijri, Gregorian
from googleapiclient.discovery import build
from google.oauth2 import service_account
import json
import re
import re
import json
import os
from langchain.docstore.document import Document
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from config import tokenizer, llm_pipeline, DEVICE



from config import tokenizer, llm_pipeline, DEVICE, SERVICE_ACCOUNT_FILE, SCOPES

Overwriting RulesForIntents.py


In [16]:
%%writefile -a RulesForIntents.py

import os
import re
import json
from langchain.docstore.document import Document
from langchain.vectorstores import FAISS
# لم نعد بحاجة لـ CharacterTextSplitter هنا لأننا سنقسم يدوياً
from config import tokenizer, llm_pipeline, DEVICE

RULES_DIR = "intent_rules"
os.makedirs(RULES_DIR, exist_ok=True)

# هذا هو الفاصل الذي سنعتمد عليه
RULE_SEPARATOR = "####_RULE_ENTRY_####"





# مسار القواعد والفاصل الأساسي
RULES_DIR = "intent_rules"
RULE_SEPARATOR = "####_RULE_ENTRY_####"

def retrieve_rule_small_to_parent(query, intent, embed_model, k=1):
    """
    دالة استرجاع القواعد المتقدمة (Small-to-Parent Strategy).
    المبدأ:
    1. تقسيم الملف إلى قواعد كاملة (Parents).
    2. تقسيم كل قاعدة إلى أجزاء صغيرة (Children).
    3. البحث في الأجزاء الصغيرة.
    4. استرجاع القاعدة الأم كاملة عند تطابق أي جزء صغير منها.
    """
    file_path = os.path.join(RULES_DIR, f"{intent}_rules.txt")
    
    if not os.path.exists(file_path):
        print(f"⚠ ملف القواعد غير موجود لـ: {intent}")
        return ""

    try:
        # 1. قراءة الملف وتقسيمه إلى قواعد كاملة (Parents)
        with open(file_path, 'r', encoding='utf-8') as f:
            full_content = f.read()
            
        # تقسيم النص بناءً على الفاصل المخصص للحصول على "القواعد الأم"
        # نستخدم الفهرس (index) كمعرف فريد للربط بين الابن والأب
        raw_parents = full_content.split(RULE_SEPARATOR)
        parent_docs = []
        
        for idx, text in enumerate(raw_parents):
            if text.strip():
                # تخزين النص الكامل مع معرف (ID) فريد
                parent_docs.append(Document(page_content=text.strip(), metadata={"parent_id": idx}))

        if not parent_docs:
            return ""

        # 2. تقسيم القواعد الأم إلى قطع صغيرة (Children)
        # حجم القطعة 150 حرفاً لضمان دقة البحث الدلالي العالية
        child_splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=30)
        
        small_docs = []
        for parent in parent_docs:
            # تقسيم نص القاعدة الواحدة إلى قطع صغيرة
            chunks = child_splitter.split_text(parent.page_content)
            for chunk in chunks:
                # ربط القطعة الصغيرة بمعرف القاعدة الأم (parent_id)
                small_docs.append(Document(page_content=chunk, metadata={"parent_id": parent.metadata["parent_id"]}))

        # 3. بناء الفهرس على القطع الصغيرة فقط (Vector Search Only)
        # هذا يضمن أننا نبحث عن أدق تفصيل في القاعدة
        vectorstore = FAISS.from_documents(small_docs, embed_model)
        retriever = vectorstore.as_retriever(search_kwargs={"k": k * 3}) # نطلب عدداً أكبر من القطع الصغيرة لضمان العثور على أفضل القواعد
        
        # تنفيذ البحث
        small_hits = retriever.invoke(query)
        
        # 4. تجميع القواعد الأم (Parents) بناءً على النتائج الصغيرة
        final_rules = []
        seen_parent_ids = set()
        
        print(f"\n🔍 [Advanced Rule Retrieval] Intent: {intent}")
        
        for hit in small_hits:
            p_id = hit.metadata["parent_id"]
            
            # إذا لم نقم بإضافة هذه القاعدة الأم من قبل
            if p_id not in seen_parent_ids:
                # استخراج النص الكامل للقاعدة الأم باستخدام المعرف
                # نبحث في قائمة parent_docs عن المستند الذي يحمل نفس الـ ID
                parent_rule = next((p for p in parent_docs if p.metadata["parent_id"] == p_id), None)
                
                if parent_rule:
                    final_rules.append(parent_rule.page_content)
                    seen_parent_ids.add(p_id)
                    
                    # طباعة توضيحية (اختياري)
                    print(f"   MATCHED PARENT RULE: ...{parent_rule.page_content[:2000]}...")
                    print(f"   -> RETRIEVED PARENT RULE ID: {p_id}")
            
            if len(final_rules) >= k:
                break
        
        # دمج القواعد المسترجعة (في حال طلبنا أكثر من قاعدة)
        result_text = "\n\n".join(final_rules)
        
        return result_text

    except Exception as e:
        print(f"⚠ خطأ في استرجاع القواعد المتقدم: {e}")
        return ""


        
def retrieve_relevant_rules(query, intent, embed_model, k=3):
    """
    دالة ذكية لاسترجاع أقرب قاعدة.
    تم التعديل: تقسيم صارم للنص لضمان عدم دمج قاعدتين في نتيجة واحدة.
    """
    file_path = os.path.join(RULES_DIR, f"{intent}_rules.txt")
    
    if not os.path.exists(file_path):
        print(f"⚠ ملف القواعد غير موجود لـ: {intent}")
        return "" 

    try:
        # 1. قراءة الملف كاملاً
        with open(file_path, 'r', encoding='utf-8') as f:
            rules_text = f.read()

        if not rules_text.strip():
            return ""

        # 2. التقسيم الصارم (Hard Split)
        # نستخدم split الخاصة بسترينج بايثون لضمان فصل كل قاعدة لوحدها
        # سواء كان الفاصل قبله سطر جديد أو بعده
        raw_texts = rules_text.split(RULE_SEPARATOR)
        
        # تنظيف النصوص (إزالة المسافات الزائدة والأسطر الفارغة)
        texts = [t.strip() for t in raw_texts if t.strip()]

        if not texts:
            return ""

        # تحويل النصوص إلى مستندات
        docs = [Document(page_content=t) for t in texts]

        # 3. إنشاء قاعدة البيانات المتجهة
        vectorstore = FAISS.from_documents(docs, embed_model)
        
        # 4. البحث (الآن k=1 ستعيد فعلاً فقرة واحدة نظيفة)
        retriever = vectorstore.as_retriever(search_kwargs={"k": k})
        relevant_docs = retriever.invoke(query)
        
        # 5. دمج النتائج (في حال طلبت k>1)
        result_rules = "\n\n".join([d.page_content for d in relevant_docs])
        
        # --- طباعة للتقييم ---
        print("\n" + "="*60)
        print(f"🔍 [DEBUG: RAG RULE RETRIEVAL - k={k}] Intent: {intent}")
        print("-" * 30)
        print(f"QUERY: {query[:1000]}...") 
        print("-" * 30)
        print("RETRIEVED RULE (Full Text):")
        # تنظيف نهائي للتأكد من عدم ظهور الفاصل في الطباعة
        clean_print = result_rules.replace(RULE_SEPARATOR, "")
        print(clean_print if clean_print else "NO RELEVANT RULE FOUND")
        print("="*60 + "\n")
        # ---------------------

        return result_rules

    except Exception as e:
        print(f"⚠ خطأ في استرجاع القواعد الذكية: {e}")
        return ""

# بقية الدوال كما هي (تأكد من إضافتها أسفل هذا الكود في الملف)
def load_intent_rules(intent):
    file_path = os.path.join(RULES_DIR, f"{intent}_rules.txt")
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read().strip()
        except Exception as e:
            print(f"⚠ خطأ في قراءة ملف القواعد لـ {intent}: {e}")
            return ""
    return ""

def deduce_administrative_rule(incoming_body, sent_body, intent):
    system_prompt = """ أنت خبير توثيق إجراءات (SOP Expert). مهمتك: تحليل الرسالة الواردة والرد المنفذ، واستخراج "بروتوكول العمل" المتبع لتعميمه مستقبلاً. استخرج البيانات التالية بدقة واختصار: 1. الهوية: (من يجب أن يتقمص البوت؟ مثال: مدير الموارد، المشرف، السكرتير). 2. الحالة: (متى نطبق هذه القاعدة؟ مثال: عند طلب إجازة، عند طلب عهده). 3. الإجراء والمخاطب: (ماذا نفعل ولمن نوجه الرد؟ مثال: الموافقة وتوجيه خطاب لمدير الإدارة المالية). 4. هيكل الرد: (كيف نبدأ وكيف نختم؟ مثال: ابدأ بـ "سعادة..."، أشر للطلب بـ "نود الإحاطة"، اختتم بـ "وتقبلوا..."). المخرجات: نص فقط، بدون مقدمات، على شكل نقاط. """
    
    user_prompt = f"""
=== الرسالة الواردة ===
{incoming_body}

=== الرد المنفذ ===
{sent_body}

استخرج بروتوكول العمل لهذا النمط:
"""

    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    
    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        output = llm_pipeline(text_input, temperature=0.01, max_new_tokens=512)[0]['generated_text'].replace(text_input, "").strip()
        return output
    except Exception as e:
        print(f"⚠ خطأ في استنتاج القاعدة: {e}")
        return None

def update_intent_rules_file(intent, new_rule):
    """
    تحديث ملف القواعد بإضافة القاعدة الجديدة مع الفاصل المخصص.
    """
    if not new_rule: return
    
    file_path = os.path.join(RULES_DIR, f"{intent}_rules.txt")
    
    try:
        existing_content = ""
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                existing_content = f.read()
        
        check_snippet = new_rule[:5000]
        if check_snippet in existing_content:
            print(f"ℹ القاعدة تبدو مكررة (موجودة مسبقاً)، لن يتم إضافتها.")
            return

        # إضافة القاعدة الجديدة مسبوقة بفاصل وسطر جديد لضمان الفصل التام
        with open(file_path, 'a', encoding='utf-8') as f:
            # نضع سطرين جديدين والفاصل لضمان أن الـ split القادم سيعمل 100%
            prefix = f"\n{RULE_SEPARATOR}\n" if existing_content else ""
            f.write(f"{prefix}{new_rule}")
            
        print(f"✅ تم تعلم قاعدة جديدة وحفظها في: {intent}_rules.txt")
        
    except Exception as e:
        print(f"⚠ خطأ في تحديث ملف القواعد: {e}")

Appending to RulesForIntents.py
